# FPL Decision Support System — Modelling Notebook
**Thesis Project** | PyCaret AutoML vs AutoKeras Deep Learning

### Pipeline Overview
1. Setup & Installs
2. Load & Prepare Data
3. PyCaret — AutoML Benchmarking
4. AutoKeras — Deep Learning
5. Model Comparison & Best Model Selection
6. Prediction Intervals (Low / Mid / High)
7. Save Predictions for Dashboard

## 1. Setup & Installs
Run this cell first. It will take a few minutes on first run.

In [ ]:
# Install dependencies
!pip install pycaret[full] -q
!pip install autokeras -q

# Mount Google Drive — your CSV should be stored here
from google.colab import drive
drive.mount('/content/drive')

print('Setup complete.')

## 2. Load & Prepare Data

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Update this path to wherever you saved the CSV in your Drive ──
DATA_PATH = '/content/drive/MyDrive/fpl_thesis/data/processed/featured_training_set.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# ── Create the TARGET: next GW points for each player ─────────────────────────
# We shift total_points by -1 within each player+season group
# so the model learns: given THIS gameweek's features, predict NEXT gameweek's points
df = df.sort_values(['name', 'season', 'GW'])
df['target'] = df.groupby(['name', 'season'])['total_points'].shift(-1)

# Drop the last GW of each season (no next GW to predict)
df = df.dropna(subset=['target']).reset_index(drop=True)
print(f'After creating target: {df.shape[0]} rows')

# ── Columns to drop before modelling ──────────────────────────────────────────
# These are identifiers, raw scores, or leakage columns
DROP_COLS = [
    'name', 'team', 'season', 'GW', 'kickoff_time',
    'total_points',   # raw target — already encoded as 'target'
    'element',        # player ID, not a feature
    'fixture',        # fixture ID, not a feature
    'modified',       # metadata
]

# Keep only columns that exist in the dataframe
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# ── Encode categorical columns ─────────────────────────────────────────────────
df['position'] = df['position'].astype('category').cat.codes   # GK=0, DEF=1, MID=2, FWD=3
df['was_home'] = df['was_home'].astype(int)

# ── Final modelling dataframe ──────────────────────────────────────────────────
model_df = df.drop(columns=DROP_COLS)

# Drop any remaining non-numeric columns
non_numeric = model_df.select_dtypes(exclude=[np.number, 'bool']).columns.tolist()
if non_numeric:
    print(f'Dropping non-numeric columns: {non_numeric}')
    model_df = model_df.drop(columns=non_numeric)

print(f'Modelling dataframe shape: {model_df.shape}')
print(f'Target stats:\n{model_df["target"].describe()}')

In [ ]:
# ── Train / Test Split ─────────────────────────────────────────────────────────
# We use the 25/26 season as the test set (most recent, unseen data)
# and 23/24 + 24/25 as training. This is a time-aware split — no random shuffle.

train_mask = df['season'].isin(['2324', '2425'])
test_mask  = df['season'] == '2526'

train_df = model_df[train_mask].reset_index(drop=True)
test_df  = model_df[test_mask].reset_index(drop=True)

# Keep player names for the test set for the final predictions output
test_meta = df[test_mask][['name', 'team', 'position', 'GW', 'season', 'value']].reset_index(drop=True)

print(f'Train: {train_df.shape} | Test: {test_df.shape}')

## 3. PyCaret — AutoML Benchmarking
PyCaret will automatically train and compare ~20 regression models using cross-validation.

In [ ]:
from pycaret.regression import *

# Initialise PyCaret experiment
pc_setup = setup(
    data            = train_df,
    target          = 'target',
    session_id      = 42,          # for reproducibility
    fold            = 5,           # 5-fold cross validation
    verbose         = True,
    use_gpu         = True,        # use T4 GPU where possible
    remove_outliers = True,        # removes extreme point outliers
    normalize       = True,        # scale features
)

print('PyCaret setup complete.')

In [ ]:
# Compare all models — this is the AutoML benchmark step
# sort by MAE (Mean Absolute Error) — most interpretable for points prediction
best_models = compare_models(
    sort     = 'MAE',
    n_select = 3,      # return top 3 models for ensemble later
    exclude  = ['ransac']  # exclude RANSAC as it struggles with this type of data
)

print('\nTop 3 models selected.')

In [ ]:
# Tune the best model further
best_model  = best_models[0]
tuned_model = tune_model(best_model, optimize='MAE', n_iter=20)

print(f'Best model after tuning: {type(tuned_model).__name__}')

In [ ]:
# Finalise — retrain on full training data (not just CV folds)
final_pycaret_model = finalize_model(tuned_model)

# Evaluate on test set
pycaret_preds = predict_model(final_pycaret_model, data=test_df)

from sklearn.metrics import mean_absolute_error, r2_score
mae_pc = mean_absolute_error(pycaret_preds['target'], pycaret_preds['prediction_label'])
r2_pc  = r2_score(pycaret_preds['target'], pycaret_preds['prediction_label'])

print(f'PyCaret Test MAE : {mae_pc:.4f}')
print(f'PyCaret Test R²  : {r2_pc:.4f}')

In [ ]:
# Save PyCaret model
save_model(final_pycaret_model, '/content/drive/MyDrive/fpl_thesis/models/pycaret_best_model')
print('PyCaret model saved.')

## 4. AutoKeras — Deep Learning
AutoKeras will automatically design and tune a neural network for the same task.

In [ ]:
import autokeras as ak
import tensorflow as tf

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {tf.config.list_physical_devices("GPU")}')

# Prepare numpy arrays for AutoKeras
X_train = train_df.drop(columns=['target']).values.astype(np.float32)
y_train = train_df['target'].values.astype(np.float32)
X_test  = test_df.drop(columns=['target']).values.astype(np.float32)
y_test  = test_df['target'].values.astype(np.float32)

In [ ]:
# AutoKeras Structured Data Regressor
# max_trials: how many different neural architectures to try
# overwrite: start fresh each time
ak_model = ak.StructuredDataRegressor(
    max_trials  = 10,
    overwrite   = True,
    seed        = 42,
    objective   = 'val_mean_absolute_error',
    directory   = '/content/drive/MyDrive/fpl_thesis/models/autokeras_trials'
)

# Train — AutoKeras handles architecture search automatically
ak_model.fit(
    X_train, y_train,
    epochs          = 50,
    validation_split= 0.1,
    verbose         = 1
)

print('AutoKeras training complete.')

In [ ]:
# Evaluate AutoKeras on test set
ak_preds_raw = ak_model.predict(X_test).flatten()

mae_ak = mean_absolute_error(y_test, ak_preds_raw)
r2_ak  = r2_score(y_test, ak_preds_raw)

print(f'AutoKeras Test MAE : {mae_ak:.4f}')
print(f'AutoKeras Test R²  : {r2_ak:.4f}')

# Export the best AutoKeras model
best_ak_model = ak_model.export_model()
best_ak_model.save('/content/drive/MyDrive/fpl_thesis/models/autokeras_best_model.keras')
print('AutoKeras model saved.')

## 5. Model Comparison
Side-by-side comparison of PyCaret best model vs AutoKeras.

In [ ]:
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    'Model'  : [f'PyCaret ({type(tuned_model).__name__})', 'AutoKeras (Neural Network)'],
    'MAE'    : [round(mae_pc, 4), round(mae_ak, 4)],
    'R²'     : [round(r2_pc, 4),  round(r2_ak, 4)],
})

print('\n===== Model Comparison =====')
print(comparison.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
comparison.plot(kind='bar', x='Model', y='MAE', ax=axes[0], color=['steelblue', 'darkorange'], legend=False)
axes[0].set_title('MAE (lower is better)')
axes[0].set_xticklabels(comparison['Model'], rotation=15, ha='right')

comparison.plot(kind='bar', x='Model', y='R²', ax=axes[1], color=['steelblue', 'darkorange'], legend=False)
axes[1].set_title('R² (higher is better)')
axes[1].set_xticklabels(comparison['Model'], rotation=15, ha='right')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/fpl_thesis/outputs/model_comparison.png', dpi=150)
plt.show()

# Select the winner
winner = 'PyCaret' if mae_pc <= mae_ak else 'AutoKeras'
print(f'\nWinner: {winner} (lower MAE)')

## 6. Prediction Intervals (Low / Mid / High)
We compute prediction intervals using all top 3 PyCaret models + AutoKeras together.
- **Mid** = mean prediction across all models
- **Low** = mean − 1 std (pessimistic scenario)
- **High** = mean + 1 std (optimistic scenario)
- **Confidence** = bucketed from the interval width

In [ ]:
# Collect predictions from all models
all_preds = []

# PyCaret top 3
for i, m in enumerate(best_models):
    fm = finalize_model(m)
    p  = predict_model(fm, data=test_df)['prediction_label'].values
    all_preds.append(p)
    print(f'Collected predictions from PyCaret model {i+1}')

# AutoKeras
all_preds.append(ak_preds_raw)
print('Collected predictions from AutoKeras')

# Stack into array: shape (n_models, n_samples)
preds_array = np.array(all_preds)   # shape: (4, n_test_rows)

mid  = preds_array.mean(axis=0)
std  = preds_array.std(axis=0)
low  = np.clip(mid - std, 0, None)  # clip at 0 — can't score negative points
high = mid + std

# Confidence: narrow interval = high confidence
interval_width = high - low
q33, q66 = np.percentile(interval_width, [33, 66])

def get_confidence(w):
    if w <= q33:   return 'High'
    elif w <= q66: return 'Medium'
    else:          return 'Low'

confidence = [get_confidence(w) for w in interval_width]

print(f'Predictions computed for {len(mid)} players/GW rows')

## 7. Save Final Predictions for Dashboard

In [ ]:
# Build the final predictions dataframe
predictions_df = test_meta.copy()
predictions_df['predicted_pts_mid']  = np.round(mid,  2)
predictions_df['predicted_pts_low']  = np.round(low,  2)
predictions_df['predicted_pts_high'] = np.round(high, 2)
predictions_df['confidence']         = confidence
predictions_df['actual_pts']         = y_test   # for validation in dashboard

# Preview
print(predictions_df[['name', 'team', 'position', 'GW', 'value',
                        'predicted_pts_low', 'predicted_pts_mid',
                        'predicted_pts_high', 'confidence', 'actual_pts']].head(10))

# Save
OUT_PATH = '/content/drive/MyDrive/fpl_thesis/data/processed/predictions.csv'
predictions_df.to_csv(OUT_PATH, index=False)
print(f'\nPredictions saved to {OUT_PATH}')
print(f'Shape: {predictions_df.shape}')

---
### ✅ Notebook Complete
The following files have been saved to your Google Drive:
- `models/pycaret_best_model` — best traditional ML model
- `models/autokeras_best_model.keras` — best deep learning model
- `outputs/model_comparison.png` — comparison chart for thesis
- `data/processed/predictions.csv` — final predictions for the dashboard

Next step: **PuLP optimisation** → squad selection from these predictions.